In [25]:
# ruff: noqa: F401

import json
from contextlib import AsyncExitStack

from backend.accounts import Account
from backend.accounts_client import read_accounts_resource
from backend.market import get_share_price
from backend.mcp_servers import researcher_mcp_servers, trader_mcp_servers
from backend.reset import reset_traders
from backend.tracers import LogTracer
from backend.traders import Trader, get_researcher, get_researcher_tool
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown

from agents import Agent, Runner, add_trace_processor, trace

load_dotenv(override=True)
console = Console()

In [2]:
get_share_price("AAPL")

333.63

### Set the MCP servers and their tools


In [3]:
servers = trader_mcp_servers() + researcher_mcp_servers("Warren")
tools_count = 0

for server in servers:
    async with server:
        tools = await server.list_tools()
        tools_count += len(tools)

console.print(f"There are {len(servers)} MCP servers, and {tools_count} tools")

There are 6 MCP servers, and 17 tools

In [4]:
async def run_research():
    async with AsyncExitStack() as stack:
        active_servers = [await stack.enter_async_context(server) for server in researcher_mcp_servers("Warren")]
        researcher = await get_researcher(active_servers, "gpt-5.4-nano")
        with trace("researcher"):
            result = await Runner.run(
                researcher,
                "What is the latest information on Amazon?",
                max_turns=30,
            )
        console.print(Markdown(result.final_output))


await run_research()

Here’s the latest publicly available update on Amazon (AMZN) based on recent sources:                              

 • Next earnings (most cited): Oct 29, 2026 (after market close) — with an EPS consensus around $1.96. (Sources:   
   EarningsRadar, EarningsToday)                                                                                   
 • What the market is watching into earnings: AWS cloud growth and margins, plus commentary around AI              
   infrastructure spend/capex and progress in AI-powered managed services. (Examples: market coverage ahead of the 
   October earnings wave that includes Amazon)                                                                     
 • AWS product/AI updates (recent months): AWS has been rolling out AI/data platform features from its 2026 Data & 
   AI / summit cycles, including SageMaker Studio 2.0, Redshift Serverless Pro, and a Generative AI Marketplace    
   (with availability windows spanning spring–fall 2026). (One compiled events recap source)                       
 • Insider activity (signal, not a catalyst): Recent reporting shows insider selling/vesting activity via SEC Form 
   144 and Form 4-style aggregators; one highlighted Form 144 example lists a restricted stock vesting and sales by
   an Amazon insider in 2026. (Sources: StockTitan Form 144 page; insider-aggregator pages)                        

If you tell me whether you care more about stock/valuation, AWS/AI, or retail/advertising, I can narrow to the most
relevant headlines and key numbers.

### Convert the researcher agent into a tool


In [6]:
researcher_tool = await get_researcher_tool(researcher_mcp_servers("Warren"), "gpt-5.4-nano")
researcher_tool.name, researcher_tool.description

('Researcher',
 "This tool researches online for news and opportunities, either based on your specific request to look into a certain stock, or generally for notable financial news and opportunities. Describe what kind of research you're looking for.")

### Create Trader Agent


In [19]:
add_trace_processor(LogTracer())
warren = Trader("Warren", model_name="gpt-5.4-nano")
await warren.run()

In [24]:
resources = await read_accounts_resource("warren")
info = json.loads(resources)
print(info["transactions"][-1])

{'symbol': 'SMH', 'quantity': 6, 'price': 397.53348, 'timestamp': '2026-10-07 12:02:12', 'rationale': 'Maintain/add to existing AI/semi momentum position. Research indicates continued upstream support (ASML/High-NA visibility, TSMC advanced-node demand, and broader semi catalysts). Technicals for SMH show strong recent momentum (MACD positive with supportive histogram), so add a small increment within cash limits to benefit from continuation while limiting risk given data access constraints.'}


In [23]:
resources

'{"name": "%3cbackend.traders.trader%20object%20at%200x115830690%3e", "balance": 10000.0, "strategy": "", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2026-10-07 12:03:41", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}'